In [1]:
from datetime import timedelta

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from pyproj import Proj, Transformer
from scipy.interpolate import griddata
import xeofs as xe
from geometry_izzyv1 import grad_sphere
from regression_izzyv1 import linregress_3D
import cartopy.crs as ccrs
import datetime

crs = ccrs.PlateCarree()
import warnings
import os
from polar_plot_utils import plot_curvilinear_data

warnings.filterwarnings('ignore')
from matplotlib.ticker import MultipleLocator

In [2]:
path = '/Users/iw2g24/PycharmProjects/SSH_project/Data/'
fig_path = '/Users/iw2g24/PycharmProjects/SSH_project/results/NPP_mca/'

ice_vel_ds = xr.open_dataset(path + 'ice_vels_no_nan_ds.nc')


In [3]:
print(ice_vel_ds)

<xarray.Dataset> Size: 2GB
Dimensions:  (time: 59, y: 2916, x: 2916)
Coordinates:
    lat      (y, x) float64 68MB ...
    lon      (y, x) float64 68MB ...
  * time     (time) datetime64[ns] 472B 2003-03-01 2003-06-01 ... 2017-09-01
Dimensions without coordinates: y, x
Data variables:
    melt     (time, y, x) float32 2GB ...


In [4]:
ann_time = pd.date_range(start='2003',end = '2017', freq='YS')
ann_time = ann_time.strftime('%Y-%m-%dT%H:%M:%S.%f000')
print(ann_time)

Index(['2003-01-01T00:00:00.000000000', '2004-01-01T00:00:00.000000000',
       '2005-01-01T00:00:00.000000000', '2006-01-01T00:00:00.000000000',
       '2007-01-01T00:00:00.000000000', '2008-01-01T00:00:00.000000000',
       '2009-01-01T00:00:00.000000000', '2010-01-01T00:00:00.000000000',
       '2011-01-01T00:00:00.000000000', '2012-01-01T00:00:00.000000000',
       '2013-01-01T00:00:00.000000000', '2014-01-01T00:00:00.000000000',
       '2015-01-01T00:00:00.000000000', '2016-01-01T00:00:00.000000000',
       '2017-01-01T00:00:00.000000000'],
      dtype='object')


In [5]:
ice_vel_ds = ice_vel_ds.assign_coords(time_ann=( ann_time))

In [6]:
print(ice_vel_ds)

<xarray.Dataset> Size: 2GB
Dimensions:   (time: 59, y: 2916, x: 2916, time_ann: 15)
Coordinates:
    lat       (y, x) float64 68MB ...
    lon       (y, x) float64 68MB ...
  * time      (time) datetime64[ns] 472B 2003-03-01 2003-06-01 ... 2017-09-01
  * time_ann  (time_ann) object 120B '2003-01-01T00:00:00.000000000' ... '201...
Dimensions without coordinates: y, x
Data variables:
    melt      (time, y, x) float32 2GB ...


In [7]:
melt_ann_mean = ice_vel_ds['melt'].groupby('time.year').mean()
print(melt_ann_mean)

<xarray.DataArray 'melt' (year: 15, y: 2916, x: 2916)> Size: 510MB
array([[[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
...
        ...,
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.]],

       [[0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        [0., 0., 0., ..., 0., 0., 0.],
        ...,
        [0., 0., 0., ..., 0., 0., 0.],


In [8]:
# Rename the 'year' dimension to 'time_ann' and assign the correct coordinates from ds['time_ann']
melt_ann_mean = melt_ann_mean.rename({'year': 'time_ann'})
melt_ann_mean = melt_ann_mean.assign_coords(time_ann=ice_vel_ds['time_ann'])
ice_vel_ds['melt_ann_mean'] = melt_ann_mean

In [9]:
print(ice_vel_ds)

<xarray.Dataset> Size: 3GB
Dimensions:        (time: 59, y: 2916, x: 2916, time_ann: 15)
Coordinates:
    lat            (y, x) float64 68MB ...
    lon            (y, x) float64 68MB ...
  * time           (time) datetime64[ns] 472B 2003-03-01 ... 2017-09-01
  * time_ann       (time_ann) object 120B '2003-01-01T00:00:00.000000000' ......
Dimensions without coordinates: y, x
Data variables:
    melt           (time, y, x) float32 2GB ...
    melt_ann_mean  (time_ann, y, x) float32 510MB 0.0 0.0 0.0 ... 0.0 0.0 0.0


In [10]:
path = '/Users/iw2g24/PycharmProjects/SSH_project/'
# ice_vels_no_nan_ds.to_netcdf(path + 'Data/ice_vels_no_nan_ds.nc')
ice_vel_ds.to_netcdf(path + 'Data/ice_vels_ds_melt_and_mean.nc')


In [17]:
print(ice_vel_ds['melt_ann_mean'].time_ann)

<xarray.DataArray 'time_ann' (time_ann: 15)> Size: 120B
array(['2003-01-01T00:00:00.000000000', '2004-01-01T00:00:00.000000000',
       '2005-01-01T00:00:00.000000000', '2006-01-01T00:00:00.000000000',
       '2007-01-01T00:00:00.000000000', '2008-01-01T00:00:00.000000000',
       '2009-01-01T00:00:00.000000000', '2010-01-01T00:00:00.000000000',
       '2011-01-01T00:00:00.000000000', '2012-01-01T00:00:00.000000000',
       '2013-01-01T00:00:00.000000000', '2014-01-01T00:00:00.000000000',
       '2015-01-01T00:00:00.000000000', '2016-01-01T00:00:00.000000000',
       '2017-01-01T00:00:00.000000000'], dtype=object)
Coordinates:
  * time_ann  (time_ann) object 120B '2003-01-01T00:00:00.000000000' ... '201...
